# Modelling: predicting the number of rings of an abalone

This notebook implements a first working model to predict `Rings` from physical measurements, following the
decisions of the EDA notebook:

- remove the 9 physically impossible records;
- one-hot encode `Sex` and standardise the numerical features, inside a scikit-learn `Pipeline`;
- evaluate with the RMSE (main metric) and the MAE, against a constant baseline (RMSE of 3.22 rings).

Every model is tracked with MLflow, so that runs can be compared in the MLflow UI and the selected model can be
reloaded from the model registry.

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from mlflow import MlflowClient
from sklearn.base import RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 500)
sns.set_theme(style="whitegrid")

# Configuration

All paths, column names and modelling constants are defined once here.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "abalone.csv"

TARGET_COL = "Rings"
CATEGORICAL_COLS = ["Sex"]
NUMERICAL_COLS = [
    "Length",
    "Diameter",
    "Height",
    "Whole weight",
    "Shucked weight",
    "Viscera weight",
    "Shell weight",
]

WEIGHT_PART_COLS = ["Shucked weight", "Viscera weight", "Shell weight"]

# Data quality threshold, justified in the EDA notebook
MAX_HEIGHT = 0.5

TEST_SIZE = 0.2
CV_FOLDS = 5
RANDOM_STATE = 42

## 1 - Data preparation

The preparation is split into small functions, so that they can be moved as is into Python modules in the next
step of the project. `get_erroneous_rows` is the function written in the EDA notebook.

In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    """Load the Abalone CSV dataset."""
    return pd.read_csv(path)


def get_erroneous_rows(df: pd.DataFrame) -> pd.Series:
    """Flag physically impossible records (zero or extreme height, inconsistent dimensions or weights)."""
    return (
        (df["Height"] <= 0)
        | (df["Height"] > MAX_HEIGHT)
        | (df["Diameter"] > df["Length"])
        | df[WEIGHT_PART_COLS].gt(df["Whole weight"], axis=0).any(axis=1)
    )


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of the dataset without its erroneous records."""
    return df.loc[~get_erroneous_rows(df)].reset_index(drop=True)


def extract_x_y(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Split the dataset into features and target."""
    return df[CATEGORICAL_COLS + NUMERICAL_COLS], df[TARGET_COL]

In [ ]:
raw_df = load_data(DATA_PATH)
df = clean_data(raw_df)
X, y = extract_x_y(df)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

print(f"Raw dataset: {len(raw_df)} rows, after cleaning: {len(df)} rows")
print(f"Train set: {len(X_train)} rows, test set: {len(X_test)} rows")
print(f"Mean rings - train: {y_train.mean():.2f}, test: {y_test.mean():.2f}")

The test set (20%) is kept aside to measure the final performance only. To compare models and choose one, a 5-fold
cross-validation on the train set is used instead: choosing the model on the test set would make its test score
optimistic.

The preprocessing is packaged with the model in a single `Pipeline`. This was the improvement suggested in the
experiment tracking course: the registered model then contains the encoder and the scaler, and can be applied
directly to raw data.

In [ ]:
def build_pipeline(regressor: RegressorMixin) -> Pipeline:
    """Build a pipeline that one-hot encodes `Sex`, standardises the measurements and applies the regressor."""
    preprocessor = ColumnTransformer(
        [
            ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
            ("numerical", StandardScaler(), NUMERICAL_COLS),
        ]
    )
    return Pipeline([("preprocessor", preprocessor), ("regressor", regressor)])


def evaluate_model(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    """Return the RMSE, MAE (in rings) and R² of a set of predictions."""
    return {
        "rmse": root_mean_squared_error(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }

**Conclusions - data preparation**

- 4,168 abalones remain after cleaning, split into 3,334 for training and 834 for testing; the mean number of rings
  is similar in both sets.
- Preprocessing and model are combined in one pipeline, and models are compared by cross-validation on the train set.

## 2 - MLflow setup

MLflow stores its runs locally, at the root of the project: run metadata in a SQLite file (`mlflow.db`) and models
in the `mlruns/` folder. Both are ignored by git, as only the code producing the experiments should be versioned.
To send the runs to a tracking server instead (for example the Docker server used in the course), set the
`MLFLOW_TRACKING_URI` environment variable before starting the notebook.

To browse the runs, launch the UI from the project root, then open [http://localhost:5000](http://localhost:5000):

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
```

In [ ]:
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")
MLFLOW_ARTIFACTS_PATH = PROJECT_ROOT / "mlruns"

DATASET_NAME = "abalone"
OBJECTIVE_NAME = "rings-regression"
EXPERIMENT_NAME = f"{DATASET_NAME}-{OBJECTIVE_NAME}"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

# Create the experiment with an explicit artifact location, otherwise MLflow stores the models next to the notebook
if client.get_experiment_by_name(EXPERIMENT_NAME) is None:
    client.create_experiment(EXPERIMENT_NAME, artifact_location=MLFLOW_ARTIFACTS_PATH.as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

Every model is trained and logged by the same function, so that all runs record the same information:

- **tags** describing the run (model family, dataset, task, experiment phase);
- **parameters**: the model hyperparameters and the data preparation settings;
- **metrics**: cross-validated RMSE on the train set (used to compare models), plus RMSE, MAE and R² on the train
  and test sets;
- the fitted **pipeline**, with an input example so that MLflow infers its input schema.

In [ ]:
RUN_ATTRIBUTES = {
    "dataset": DATASET_NAME,
    "objective": OBJECTIVE_NAME,
    "task": "regression",
}
DATA_PARAMS = {
    "train_set_size": len(X_train),
    "test_set_size": len(X_test),
    "test_size": TEST_SIZE,
    "cv_folds": CV_FOLDS,
    "random_state": RANDOM_STATE,
    "max_height": MAX_HEIGHT,
}


def train_and_log_model(
    regressor: RegressorMixin,
    model_family: str,
    experiment_phase: str,
    params: dict,
    skops_trusted_types: list[str] | None = None,
) -> dict:
    """Train a pipeline, evaluate it and log the run (tags, parameters, metrics, model) to MLflow.

    `skops_trusted_types` lists the classes MLflow may serialise beyond the ones it trusts by default.
    """
    run_timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    params_suffix = "-".join(f"{key}-{value}" for key, value in params.items())
    run_name = "-".join(filter(None, [model_family, params_suffix, run_timestamp]))

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({**RUN_ATTRIBUTES, "model_family": model_family, "experiment_phase": experiment_phase})
        mlflow.log_params({**params, **DATA_PARAMS})

        pipeline = build_pipeline(regressor)
        cv_rmse = -cross_val_score(
            pipeline, X_train, y_train, cv=CV_FOLDS, scoring="neg_root_mean_squared_error"
        ).mean()
        pipeline.fit(X_train, y_train)

        train_metrics = evaluate_model(y_train, pipeline.predict(X_train))
        test_metrics = evaluate_model(y_test, pipeline.predict(X_test))
        metrics = {
            "cv_rmse": cv_rmse,
            **{f"train_{name}": value for name, value in train_metrics.items()},
            **{f"test_{name}": value for name, value in test_metrics.items()},
        }
        mlflow.log_metrics(metrics)

        model_info = mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="model",
            input_example=X_train.head(),
            skops_trusted_types=skops_trusted_types,
        )

    return {"model_family": model_family, **params, **metrics, "model_uri": model_info.model_uri}

**Conclusions - MLflow setup**

- All runs go to the `abalone-rings-regression` experiment, stored locally and ignored by git.
- One function trains, evaluates and logs every model, so all runs are comparable column by column in the UI.

## 3 - First models: baseline and linear regression

The baseline always predicts the mean number of rings of the train set: any model must beat it to be useful.
The linear regression is the first version of the model required for this project.

In [ ]:
results = [
    train_and_log_model(
        DummyRegressor(strategy="mean"),
        model_family="baseline",
        experiment_phase="baseline",
        params={},
    ),
    train_and_log_model(
        LinearRegression(),
        model_family="linear_regression",
        experiment_phase="first_model",
        params={},
    ),
]

METRIC_COLS = ["cv_rmse", "train_rmse", "test_rmse", "test_mae", "test_r2"]
pd.DataFrame(results).set_index("model_family")[METRIC_COLS].round(3)

A linear model on collinear features can have unstable coefficients (see the EDA). Look at the fitted coefficients
to check it, keeping in mind that they apply to standardised features.

In [ ]:
linear_model = mlflow.sklearn.load_model(results[1]["model_uri"])
pd.Series(
    linear_model.named_steps["regressor"].coef_,
    index=linear_model.named_steps["preprocessor"].get_feature_names_out(),
    name="coefficient",
).round(2)

The coefficients confirm the collinearity issue: `Whole weight` (+4.5) and `Shucked weight` (-4.4) largely offset
each other, and `Length` even gets a negative coefficient although it is positively correlated with `Rings` on its
own. They still make sense
when read together: at a given total weight, an abalone with more meat tends to be younger, which is the
"proportion of meat versus shell" signal mentioned in the EDA.

**Conclusions - first models**

- The baseline reaches a cross-validated RMSE of 3.20 rings, consistent with the EDA (3.22).
- The linear regression brings the RMSE down to 2.19 rings in cross-validation (2.14 on the test set) and explains
  58% of the variance: on average, it is wrong by about 1.6 rings (1.6 years).
- Train and cross-validation RMSE are identical: the model does not overfit, it is limited by its linearity.
- Its coefficients are unstable because of collinearity, which motivates trying Ridge, and the curved relationships
  seen in the EDA motivate trying a non-linear model.

## 4 - Comparison with a non-linear model

The EDA showed curved relationships between the weights and `Rings`. A random forest captures them without any
feature engineering, which measures how much the linear model loses by being linear. It is used as a point of
comparison only: the project guidelines state that a simple linear regression is enough.

A random forest with fully grown trees memorises the training set, so the trees are regularised by requiring at
least 5 abalones per leaf. MLflow refuses by default to save tree structures, as a malicious file could exploit them
when loaded; since this model is created here, its tree class is explicitly declared as trusted.

In [ ]:
RANDOM_FOREST_PARAMS = {"n_estimators": 300, "min_samples_leaf": 5}

results.append(
    train_and_log_model(
        RandomForestRegressor(**RANDOM_FOREST_PARAMS, random_state=RANDOM_STATE, n_jobs=-1),
        model_family="random_forest",
        experiment_phase="comparison",
        params=RANDOM_FOREST_PARAMS,
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )
)

results_df = pd.DataFrame(results).set_index("model_family")
results_df[METRIC_COLS].round(3)

## 5 - Model selection

Models are compared on the cross-validated RMSE. The test set is only used afterwards, to report the performance of
the selected model.

In [ ]:
linear_cv_rmse = results_df.loc["linear_regression", "cv_rmse"]
forest_cv_rmse = results_df.loc["random_forest", "cv_rmse"]
print(f"Random forest gain over the linear regression: {linear_cv_rmse - forest_cv_rmse:.3f} rings of RMSE")
print(f"Relative gain: {(linear_cv_rmse - forest_cv_rmse) / linear_cv_rmse:.1%}")

The random forest is only marginally better than the linear regression in cross-validation (0.05 ring, about 2%),
and both reach the same RMSE on the test set (2.14). It is also much heavier (300 trees against 10 coefficients),
slower and harder to explain, and its training RMSE (1.50) is well below its cross-validated RMSE, a sign that it
partly memorises the training set even with regularised trees.

The **linear regression is selected**: for a practically identical error, it is simpler to train, to serve in the
API and to explain.

In [ ]:
SELECTED_MODEL_FAMILY = "linear_regression"
selected_model_uri = results_df.loc[SELECTED_MODEL_FAMILY, "model_uri"]

## 6 - Model registry

As in the experiment tracking course, the selected model is registered under a name describing its purpose rather
than its algorithm, so that a better model can replace the linear regression later without changing the code that
uses it. The `champion` alias points to the version currently used.

In [ ]:
REGISTERED_MODEL_NAME = f"{DATASET_NAME}-rings-regressor"

registered_model_version = mlflow.register_model(model_uri=selected_model_uri, name=REGISTERED_MODEL_NAME)
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="champion",
    version=registered_model_version.version,
)

champion = client.get_model_version_by_alias(name=REGISTERED_MODEL_NAME, alias="champion")
print(f"Model: {champion.name}, champion version: {champion.version}")

The champion is loaded through its alias. Since the preprocessing is part of the pipeline, it is applied directly to
raw measurements.

In [ ]:
champion_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}@champion")

X_test.head().assign(
    true_rings=y_test.head(),
    predicted_rings=champion_model.predict(X_test.head()).round(1),
)

## 7 - Error analysis

The EDA predicted that the error would grow with the age, since size stops being informative once growth slows
down. Check it on the test set.

In [ ]:
test_predictions = champion_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(x=y_test, y=test_predictions, alpha=0.3, s=15, ax=ax)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="tab:red", linestyle="--")
ax.set_xlabel("True rings")
ax.set_ylabel("Predicted rings")
ax.set_title("Linear regression - test set")
plt.show()

In [ ]:
age_groups = pd.cut(y_test, bins=[0, 7, 10, 13, 30], labels=["1-7", "8-10", "11-13", "14+"])
errors = pd.DataFrame({"error": test_predictions - y_test, "absolute_error": (test_predictions - y_test).abs()})

errors.groupby(age_groups, observed=True).agg(
    count=("error", "size"),
    mean_error=("error", "mean"),
    mae=("absolute_error", "mean"),
).round(2)

The errors follow the pattern announced by the EDA. Young abalones are overestimated by about 1 ring, and old
abalones (14 rings or more) are underestimated by 3.4 rings on average: the model pulls its predictions towards the
mean age because, beyond a certain size, the measurements no longer distinguish a 12-year-old abalone from a
20-year-old one. The average error stays around 1.2 to 1.4 rings below 14 rings, but reaches 3.6 rings for the
oldest abalones.

**Conclusions - error analysis**

- The model is reliable for young and middle-aged abalones (about 1.3 rings of average error), which represent 87% of
  the test set.
- It systematically underestimates old abalones. This limit comes from the data rather than from the model choice:
  the random forest reaches the same test error.

## 8 - Conclusions

- **Data**: 9 physically impossible records removed (0.2%); `Sex` one-hot encoded and measurements standardised,
  inside a scikit-learn pipeline stored with the model.
- **Performance**: the linear regression reaches an RMSE of 2.19 rings in cross-validation and 2.14 on the test set
  (MAE of 1.6 rings, R² of 0.58), against 3.2 for a constant baseline.
- **Model choice**: a regularised random forest does not do better on the test set and only gains 2% in
  cross-validation; the linear regression is kept for its simplicity, as allowed by the project guidelines.
- **Limits**: old abalones are underestimated by more than 3 rings on average, because size stops being informative
  once growth slows down.
- **Tracking**: every run (baseline, linear regression, random forest) is logged in the `abalone-rings-regression`
  MLflow experiment, and the selected pipeline is registered as `abalone-rings-regressor` with the `champion` alias.

**Next steps**: the data preparation, training and evaluation functions of this notebook are moved into
`src/modelling/` and orchestrated with Prefect (see the README). Possible model improvements, out of scope for this first version: log-transformed
weights or weight ratios as features, and a log-transformed target to better handle old abalones.